# **Netflix Content Clustering and Recommendation System**
**Project Type:** Exploratory Data Analysis (EDA) + Unsupervised Machine Learning  
**Contribution:** Individual Project

## Project Summary
This project explores the Netflix Movies and TV Shows catalogue and applies text-based clustering to group titles with similar content. The analysis begins by inspecting the dataset, understanding its dimensions and variables, and examining content types, ratings, and genres. Missing values in descriptive fields are handled so that the dataset can be used consistently in later analysis.

For the machine-learning stage, information from the director, cast, country, genre/category (`listed_in`), and description fields is combined into a single text representation. TF-IDF converts this combined text into numerical features, giving greater weight to informative terms while reducing the influence of very common words. K-Means then groups titles into clusters based on their text features. A sampled cosine silhouette score is used to assess cluster separation without requiring the full sparse matrix to be evaluated at once.

The resulting clusters are inspected through title examples, genre summaries, and cluster-size summaries. Finally, a recommendation function retrieves titles from the same cluster as a selected title, offering a simple content-based discovery experience. The saved model artifacts and processed output support reuse of the analysis.

The project demonstrates how catalogue metadata can help organise a large streaming library, identify groups of related content, and support content discovery. Cluster membership should be interpreted as similarity in the available metadata rather than a guarantee that two titles have the same audience or quality. The usefulness of recommendations depends on the completeness and quality of the metadata and on the final cluster structure.

# **GitHub Link**
Add the URL of your project repository here after uploading the notebook and any required project files.

# **Problem Statement**
Netflix contains a large catalogue of movies and TV shows. Users may find it difficult to discover relevant titles when browsing a catalogue at scale. The goal of this project is to explore the catalogue and group titles with similar metadata and descriptions using unsupervised machine learning.

#### **Business Objective**
Build an interpretable prototype that:
- describes the composition of the Netflix catalogue using exploratory analysis;
- preprocesses text metadata into machine-readable features;
- clusters titles using K-Means and evaluates the cluster configuration;
- interprets clusters using representative titles and genre summaries; and
- demonstrates a simple content-based recommendation function.

# **General Guidelines**
- Run the notebook from top to bottom in the same environment.
- Confirm the dataset path before running the loading cells.
- Use charts and printed summaries to support conclusions rather than making unsupported assumptions.
- Cluster labels are arbitrary identifiers; they do not indicate quality or ranking.
- Results may vary if the dataset, preprocessing, or model settings change.

# ***Let's Begin!***

## ***1. Know Your Data***
This section loads the Netflix catalogue and inspects its structure, dimensions, columns, data types, and missing values.

### Import Libraries
Import the libraries required for loading, inspecting, and analysing the dataset.

In [1]:
import pandas as pd

### Dataset Loading
Load the Netflix titles CSV. Make sure the relative path points to your local dataset.

In [3]:
df = pd.read_csv("../data/netflix_titles.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../data/netflix_titles.csv'

### Dataset First View
Preview the first few records to understand how each title is represented.

In [ ]:
df.head()

### Dataset Rows & Columns Count
Check how many records and attributes are available.

In [ ]:
df.shape


### Dataset Columns
Inspect the column names supplied with the dataset.

In [ ]:
df.columns

### Dataset Information
Review data types and non-null counts.

In [ ]:
df.info()

### Quick Combined Inspection
This cell repeats some of the initial checks in one place.

In [ ]:
df.head()
df.shape
df.info()

### Missing Values / Null Values
Measure missingness before deciding how to handle it.

In [ ]:
df.isnull().sum()

### Missing-Value Handling: Director

In [ ]:
df['director'] = df['director'].fillna('Unknown')

### Missing-Value Handling: Cast

In [ ]:
df['cast'] = df['cast'].fillna('Unknown')

### Missing-Value Handling: Country

In [ ]:
df['country'] = df['country'].fillna('Unknown')

### Missing-Value Handling: Date Added

In [ ]:
df['date_added'] = df['date_added'].fillna('Unknown')

### Missing-Value Handling: Rating

In [ ]:
df['rating'] = df['rating'].fillna('Unknown')

### Missing-Value Handling: Duration

In [ ]:
df['duration'] = df['duration'].fillna('Unknown')

### Catalogue Composition: Content Type
Count movies and TV shows to understand the catalogue mix.

In [ ]:
df['type'].value_counts()

## ***2. Understanding Your Variables***
### Import Plotting Libraries
The following cells explore content type, ratings, and genres using charts and frequency tables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### Content-Type Frequency

In [ ]:
df['type'].value_counts()

#### Chart 1 — Movies vs TV Shows
**Why this chart?** A count plot gives a direct comparison of the two content types.  
**What to inspect:** Which type appears more frequently, and how large is the difference?  
**Business relevance:** Catalogue mix can inform browsing categories and content-planning discussions.

In [ ]:
sns.countplot(data = df , x = 'type')

plt.title('Movies vs TV Shows')
plt.xlabel('Type')
plt.ylabel('Number of Titles')

plt.show()

### Rating Frequency

In [ ]:
df['rating'].value_counts()

#### Chart 2 — Distribution of Netflix Ratings
**Why this chart?** A bar chart makes the most common maturity/content ratings easy to compare.  
**What to inspect:** Identify the most frequent ratings and any rare categories.  
**Business relevance:** Rating distributions can inform audience filters and catalogue presentation.

In [ ]:
plt.figure(figsize = (10 , 5))

sns.countplot(data = df , x = 'rating' , order = df['rating'].value_counts().index)

plt.title('Distribution of Netflix Ratings')
plt.xlabel('Rating')
plt.ylabel('Number of Titles')
plt.xticks(rotation = 45)
plt.show()

## ***3. Data Wrangling and Genre Exploration***
The next cells reload the source data for the modelling workflow and inspect its structure before preprocessing.

In [ ]:
import pandas as pd

### Reload Dataset for Modelling
The modelling workflow starts from the source CSV.

In [ ]:
df = pd.read_csv("../data/netflix_titles.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())

### Standardise Missing Metadata
Missing descriptive fields are replaced with consistent placeholder values where appropriate.

In [ ]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['date_added'] = df['date_added'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Unknown')
df['duration'] = df['duration'].fillna('Unknown')
df['description'] = df['description'].fillna('')
df['listed_in'] = df['listed_in'].fillna('Unknown')

print("Missing values handled.")
display(df.isnull().sum())

### Split Genre Labels
Expand the multi-genre field into individual genre entries for frequency analysis.

In [ ]:
genre_list = []

for genres in df['listed_in'].str.split(', '):
    genre_list.extend(genres)

### Most Common Genres — Frequency Table

In [ ]:
genre_counts = pd.Series(genre_list).value_counts()
genre_counts.head(15)

#### Chart 3 — Top 15 Netflix Genres
**Why this chart?** A sorted bar chart highlights the genres that occur most often.  
**What to inspect:** Compare the most frequent genres and note the long tail of less common genres.  
**Business relevance:** Genre frequency can guide catalogue navigation and help describe the content mix.

In [ ]:
plt.figure(figsize = (12 , 6))

genre_counts.head(15).plot(kind = 'bar')
plt.title('Top 15 Netflix Genres')
plt.xlabel('Genre')
plt.ylabel('Number of Titles')
plt.xticks(rotation = 45 , ha = 'right')
plt.tight_layout()
plt.show()

### Feature Engineering: Combined Text
Combine director, cast, country, genres, and description into one text field. This creates a unified representation for text vectorisation.

In [ ]:
feature_columns = ['director', 'cast', 'country', 'listed_in', 'description']

df[feature_columns] = df[feature_columns].fillna('').astype(str)

df['combined_features'] = (
    df['director'] + ' ' +
    df['cast'] + ' ' +
    df['country'] + ' ' +
    df['listed_in'] + ' ' +
    df['description']
)

### Preview Engineered Features
Check a few titles to verify the combined text field.

In [ ]:
df[['title', 'combined_features']].head()

## ***4. Text Preprocessing and Feature Engineering***
### TF-IDF Vectorisation
TF-IDF transforms the combined text into a sparse numeric matrix. English stop words are excluded to reduce the contribution of common terms.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("Scikit-learn is working!")


### Create the TF-IDF Matrix
The matrix shape indicates the number of titles and vocabulary features.

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(tfidf_matrix.shape)

## ***5. Unsupervised Learning: K-Means Clustering***
K-Means groups titles based on their TF-IDF feature vectors. The following cells fit an initial model and inspect the assigned clusters.

In [ ]:
from sklearn.cluster import KMeans

### Configure the Initial K-Means Model

In [ ]:
kmeans = KMeans(n_clusters = 10 , random_state = 42 , n_init = 10)

### Fit the Initial Model

In [ ]:
kmeans.fit(tfidf_matrix)

### Confirm Clustering Completion

In [ ]:
print("Clustering completed!")

### Add Cluster Labels to the Dataset

In [ ]:
df['cluster'] = kmeans.labels_

### Preview Title-to-Cluster Assignments

In [ ]:
df[['title', 'cluster']].head(10)

### Cluster Size Distribution

In [ ]:
df['cluster'].value_counts().sort_index()

### Representative Titles by Cluster
Review sample titles from each cluster to get an initial sense of the groupings. The printed examples are illustrative, not a complete description of a cluster.

In [ ]:
for cluster in range(10):
    print(f"\n=======CLUSTER {cluster}=======")
    print(df[df['cluster'] == cluster]['title'].head(10).to_string(index = False))

### Genre-Level Interpretation of Initial Clusters
Use the genre metadata to help describe what each cluster contains.

In [ ]:
for cluster in range(10):
    print(f"\n========CLUSTER {cluster}========")
    cluster_data = df[df['cluster'] == cluster]
    genres = []

    for genre_list in cluster_data['listed_in']:
        genres.extend(genre_list.split(', '))
    print(pd.Series(genres).value_counts().head(5))

## ***6. Cluster Evaluation and Model Selection***
A silhouette score estimates how well-separated the clusters are. Because TF-IDF is sparse and the catalogue can be large, the notebook evaluates a reproducible sample using cosine distance.

In [ ]:
df['cluster'].value_counts().sort_index()

### Evaluate a Sample of the TF-IDF Matrix

In [ ]:
from sklearn.metrics import silhouette_score

sample_size = min(2000, tfidf_matrix.shape[0])
sample_indices = df.sample(n=sample_size, random_state=42).index.to_numpy()

sample_matrix = tfidf_matrix[sample_indices]
sample_labels = df.loc[sample_indices, 'cluster'].to_numpy()

score = silhouette_score(
    sample_matrix,
    sample_labels,
    metric='cosine'
)

print(f"Preliminary 10-cluster Silhouette Score (Cosine): {score:.4f}")


### Prepare a Reproducible Evaluation Sample

In [ ]:
sample_size = min(2000, tfidf_matrix.shape[0])

sample_indices = df.sample(
    n=sample_size,
    random_state=42
).index.to_numpy()

sample_matrix = tfidf_matrix[sample_indices]

### Compare Candidate Values of K
Fit K-Means for several cluster counts and calculate the sampled cosine silhouette score.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

k_values = [3, 4, 5, 6, 7, 8, 9, 10]
silhouette_scores = []

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(sample_matrix)

    score = silhouette_score(
        sample_matrix,
        labels,
        metric='cosine'
    )

    silhouette_scores.append(score)
    print(f"K = {k}, Silhouette Score = {score:.4f}")

#### Chart 4 — Silhouette Score by Number of Clusters
**Why this chart?** It compares candidate cluster counts on one scale.  
**What to inspect:** Higher silhouette scores generally indicate better separation, but the score should be considered alongside interpretability and cluster size.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(k_values, silhouette_scores, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score (Cosine)')
plt.title('Silhouette Score vs Number of Clusters')
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Fit the Final K-Means Configuration
The notebook uses five clusters in the final model below. This choice should be reviewed alongside the candidate scores and the usefulness of the resulting groups.

In [ ]:
kmeans = KMeans(n_clusters = 5 , random_state = 42 , n_init = 10)
df['cluster'] = kmeans.fit_predict(tfidf_matrix)

### Final Sampled Silhouette Evaluation
This score evaluates the final model on the sampled titles using cosine distance.

In [ ]:
final_sample_labels = kmeans.predict(sample_matrix)

final_score = silhouette_score(
    sample_matrix,
    final_sample_labels,
    metric='cosine'
)

print(f"Final 5-cluster Silhouette Score (Cosine): {final_score:.4f}")

## ***7. Cluster Analysis and Interpretation***
Summarise the number and percentage of titles in each final cluster, then inspect representative metadata to give the groups practical meaning.

In [ ]:
cluster_counts = df['cluster'].value_counts().sort_index()

cluster_summary = pd.DataFrame({
    'Cluster': cluster_counts.index,
    'Number of Titles': cluster_counts.values,
    'Percentage': (cluster_counts.values / len(df) * 100).round(2)
})

display(cluster_summary)

### Representative Titles in Final Clusters

In [ ]:
for cluster in sorted(df['cluster'].unique()):
    cluster_data = df[df['cluster'] == cluster]

    print(f"\n===== CLUSTER {cluster} =====")
    print(f"Total titles: {len(cluster_data)}")

    print("\nTop genres:")
    genres = cluster_data['listed_in'].dropna().str.split(', ').explode()
    print(genres.value_counts().head(5))

    print("\nSample titles:")
    print(
        cluster_data[['title', 'type', 'listed_in']]
        .sample(n=min(10, len(cluster_data)), random_state=42)
        .to_string(index=False)
    )

### Top Genres in Final Clusters

In [ ]:
for cluster in sorted(df['cluster'].unique()):
    print(f"\n======== CLUSTER {cluster} ========")

    cluster_data = df[df['cluster'] == cluster]
    genres = cluster_data['listed_in'].dropna().str.split(', ').explode()

    print(f"Number of titles: {len(cluster_data)}")
    print("\nTop 10 genres:")
    print(genres.value_counts().head(10))

## ***8. Content-Based Recommendation Demonstration***
Cosine similarity can compare TF-IDF vectors. The recommendation function below uses cluster membership to retrieve related catalogue titles; it is a simple baseline rather than a fully ranked similarity recommender.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

### Check the Cluster of a Sample Title

In [ ]:
df[df['title'].str.lower() == 'kota factory'][['title', 'cluster']]

### Define the Recommendation Function
The function checks whether a title exists and returns titles from its assigned cluster.

In [ ]:
def recommend_by_cluster(title, n=10):
    matches = df[df['title'].str.casefold() == title.casefold()]

    if matches.empty:
        return f"Title not found: {title}"

    index = matches.index[0]
    cluster = df.loc[index, 'cluster']
    cluster_indices = df.index[df['cluster'] == cluster].to_numpy()

    similarity_scores = cosine_similarity(
        tfidf_matrix[index],
        tfidf_matrix[cluster_indices]
    ).flatten()

    ranked_positions = similarity_scores.argsort()[::-1]

    ranked_positions = [
        i for i in ranked_positions
        if cluster_indices[i] != index
    ][:n]

    recommended_indices = cluster_indices[ranked_positions]

    result = df.loc[
        recommended_indices,
        ['title', 'type', 'release_year', 'listed_in']
    ].copy()

    result['similarity'] = similarity_scores[ranked_positions]

    return result.reset_index(drop=True)

### Example Recommendation — Kota Factory

In [ ]:
recommend_by_cluster("Kota Factory")

### Example Recommendation — Squid Game

In [ ]:
recommend_by_cluster("Squid Game")

### Example Recommendation — The Crowned Clown

In [ ]:
recommend_by_cluster("The Crowned Clown")

### Example Recommendation — Naruto

In [ ]:
recommend_by_cluster("Naruto")

## ***9. Save Artifacts and Check Reproducibility***
Save the vectoriser, TF-IDF matrix, fitted model, and processed dataset so the workflow can be reused. Confirm that output paths exist in the environment before relying on the saved files.

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
os.makedirs("../output", exist_ok=True)

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, "../models/tfidf_matrix.pkl")
joblib.dump(kmeans, "../models/kmeans.pkl")

df.to_csv("../output/netflix_processed.csv", index=False)

print("All files saved successfully!")

### Verify Saved Artifact Files

In [ ]:
files = [
    "../models/tfidf_vectorizer.pkl",
    "../models/tfidf_matrix.pkl",
    "../models/kmeans.pkl",
    "../output/netflix_processed.csv"
]

for file in files:
    print(file, "→", os.path.getsize(file), "bytes")

### Edge Case — Unknown Title
Check that the recommendation function handles a title that is not in the dataset.

In [ ]:
result = recommend_by_cluster("Nonexistent Netflix Title")

if isinstance(result, str):
    print(result)
else:
    display(result)


## ***10. Solution to the Business Objective***
The workflow provides an initial catalogue exploration, text-based grouping, cluster summaries, and a demonstration recommendation function. To use this as a product feature, validate recommendation relevance with user feedback or offline ranking metrics, and compare it with a direct cosine-similarity ranking approach.

# **Conclusion**
This project combines exploratory data analysis with TF-IDF-based text representation and K-Means clustering to organise Netflix titles by similarities in their metadata and descriptions. The EDA charts describe catalogue composition and genre frequency, while cluster-size summaries, example titles, and genre summaries help interpret the model output. The recommendation examples demonstrate how cluster membership can support basic content discovery.

The approach is a useful prototype, but it has limitations: it depends on metadata quality, K-Means imposes a fixed number of clusters, and titles grouped together are not necessarily similar in viewer preference. Future improvements could compare a second clustering algorithm, test alternative values of K more systematically, rank candidates by cosine similarity within or across clusters, and evaluate recommendations with human judgement or user-interaction data.

### ***Notebook Complete — Re-run All Cells Before Submission***
Before submitting, run the notebook from top to bottom, confirm the CSV path and output directories, inspect all charts and printed summaries, and replace the GitHub placeholder with your repository URL.